# Day 02d: Attention — How Tokens Talk to Each Other

**Time:** ~20 minutes  
**Prerequisites:** Day 02c (Embeddings)  
**Goal:** Build self-attention from scratch using real GPT-2 weights.

---

We now have a `(seq_len, 768)` matrix — one vector per token. But each token's vector was created **independently** (just a lookup + position add). The tokens don't know about each other yet.

Attention fixes this. It lets every token **look at every previous token** and decide which ones are relevant to its prediction.

## The Analogy: Customer Support Ticket Routing

Imagine you're a customer support agent handling a ticket. You can see the full ticket text, but you need to decide which parts matter for your response.

```
Ticket: "My GPU server crashed after the CUDA update and now inference is slow"

If you're resolving "slow":
  - You look at "GPU" (hardware context)          — high attention
  - You look at "inference" (what's slow)          — high attention  
  - You look at "CUDA update" (possible cause)     — high attention
  - You look at "My" (not very useful)             — low attention
  - You look at "and" (grammar, irrelevant)        — low attention
```

Attention does exactly this — for every token position, it computes a relevance score against every previous position, then takes a weighted mix of their information.

## Setup: Load Weights, Tokenizer, and Embed

All the code from notebooks 02a-02c.

In [ ]:
import json, struct, os, re
import numpy as np

MODEL_DIR = "gpt2_weights"

# --- Load weights ---
def parse_safetensors(path):
    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(header_len))
        data_start = 8 + header_len
        tensors = {}
        for name, info in header.items():
            if name == "__metadata__": continue
            dtype = {"F32": np.float32, "F16": np.float16}[info["dtype"]]
            start, end = info["data_offsets"]
            f.seek(data_start + start)
            tensors[name] = np.frombuffer(f.read(end-start), dtype=dtype).reshape(info["shape"])
        return tensors

weights = parse_safetensors(os.path.join(MODEL_DIR, "model.safetensors"))

# --- Tokenizer ---
with open(os.path.join(MODEL_DIR, "vocab.json")) as f: vocab = json.load(f)
with open(os.path.join(MODEL_DIR, "merges.txt")) as f: merges_raw = f.read().strip().split("\n")[1:]
id_to_token = {v: k for k, v in vocab.items()}
bpe_ranks = {tuple(m.split()): i for i, m in enumerate(merges_raw)}
def bytes_to_unicode():
    bs = list(range(33,127))+list(range(161,173))+list(range(174,256))
    cs = bs[:]; n=0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256+n); n+=1
    return dict(zip(bs,[chr(c) for c in cs]))
byte_encoder = bytes_to_unicode()
byte_decoder = {v:k for k,v in byte_encoder.items()}
GPT2_PAT = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\s\w]+|\s+(?!\S)|\s+""")
def get_pairs(t): return set(zip(t[:-1],t[1:]))
def bpe_merge(tokens):
    while True:
        pairs=get_pairs(tokens)
        if not pairs: break
        best=min(pairs,key=lambda p:bpe_ranks.get(p,float("inf")))
        if best not in bpe_ranks: break
        a,b=best; new=[]; i=0
        while i<len(tokens):
            if i<len(tokens)-1 and tokens[i]==a and tokens[i+1]==b: new.append(a+b); i+=2
            else: new.append(tokens[i]); i+=1
        tokens=new
    return tokens
def encode(text):
    ids=[]
    for chunk in re.findall(GPT2_PAT,text):
        uc=[byte_encoder[b] for b in chunk.encode("utf-8")]
        for t in bpe_merge(uc): ids.append(vocab[t])
    return ids
def decode(ids): 
    t="".join(id_to_token[i] for i in ids)
    return bytes([byte_decoder[c] for c in t]).decode("utf-8",errors="replace")

# --- Embed ---
wte = weights["wte.weight"]
wpe = weights["wpe.weight"]
def embed(token_ids):
    return wte[token_ids] + wpe[:len(token_ids)]

print(f"Ready. Weights: {len(weights)} tensors, Vocab: {len(vocab):,} tokens")

## Step 1: What Are Q, K, V?

Attention works like a **key-value database query**:

```
Q (Query):  "What am I looking for?"
K (Key):    "What can I be found by?" 
V (Value):  "What information do I contain?"
```

It's like a Redis `GET` — except instead of exact key match, you get a **weighted blend** of all values based on how well each key matches your query.

```
Database analogy:
┌─────────────────────────────────────────────┐
│  SQL:    SELECT value                       │
│          FROM kv_store                       │
│          WHERE key LIKE query                │
│          (returns ONE exact match)            │
│                                              │
│  Attention: result = weighted_avg(           │
│               values,                        │
│               weights=softmax(query . keys)) │
│          (returns BLEND of all values)        │
└─────────────────────────────────────────────┘
```

Every token produces a Q, K, and V vector. GPT-2 computes all three with a **single matmul** using a combined weight matrix `c_attn.weight` of shape `[768, 2304]` (that's `768 x 3*768`).

In [ ]:
# Get the attention weights from block 0
c_attn_W = weights["h.0.attn.c_attn.weight"]  # (768, 2304)
c_attn_b = weights["h.0.attn.c_attn.bias"]    # (2304,)

print(f"c_attn.weight shape: {c_attn_W.shape}")
print(f"  That's ({c_attn_W.shape[0]}, 3 x {c_attn_W.shape[0]}) = (d_model, 3 x d_model)")
print(f"  It contains Q, K, V projections stacked side by side:")
print()
print(f"  ┌──── 768 ────┬──── 768 ────┬──── 768 ────┐")
print(f"  │    W_Q      │    W_K      │    W_V      │  768 rows")
print(f"  └─────────────┴─────────────┴─────────────┘")
print(f"  └──────────────── 2304 ────────────────────┘")
print()

# Compute Q, K, V for our sentence
text = "The cat sat"
token_ids = encode(text)
x = embed(token_ids)  # (3, 768)

# One matmul to get all three
qkv = x @ c_attn_W + c_attn_b  # (3, 2304)

# Split into Q, K, V
d_model = 768
Q = qkv[:, :d_model]           # (3, 768)
K = qkv[:, d_model:2*d_model]  # (3, 768)
V = qkv[:, 2*d_model:]         # (3, 768)

print(f"Input x shape:  {x.shape}   (seq_len=3, d_model=768)")
print(f"QKV combined:   {qkv.shape} (seq_len=3, 3*d_model=2304)")
print(f"Q shape:        {Q.shape}   (seq_len=3, d_model=768)")
print(f"K shape:        {K.shape}   (seq_len=3, d_model=768)")
print(f"V shape:        {V.shape}   (seq_len=3, d_model=768)")

## Step 2: Scaled Dot-Product Attention

Now we compute how much each token attends to every other token.

The formula (Book Figure 2.8):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Step by step:

```
1. Scores = Q x K^T          How well does each query match each key?
                               Shape: (seq_len, seq_len) — every pair

2. Scale  = Scores / sqrt(dk) Normalize so values don't explode
                               Like normalizing metrics before comparing

3. Mask   = apply causal mask  Can't attend to future tokens
                               Upper triangle = -infinity

4. Weights = softmax(Scores)   Convert to probabilities (sum to 1 per row)
                               Like a load balancer distributing traffic

5. Output = Weights x V        Weighted blend of value vectors
```

In [ ]:
def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

# Let's do it step by step with FULL Q, K, V (single head for now)
seq_len = Q.shape[0]  # 3
d_k = Q.shape[1]      # 768

# Step 1: Raw attention scores
scores = Q @ K.T  # (3, 3)
print(f"Step 1 — Raw scores (Q x K^T):")
print(f"  Shape: {scores.shape} — every token vs every token")
print(f"  Values:\n{scores.round(1)}")
print()

# Step 2: Scale
scores_scaled = scores / np.sqrt(d_k)
print(f"Step 2 — Scaled by 1/sqrt({d_k}) = 1/{np.sqrt(d_k):.1f}:")
print(f"  Values:\n{scores_scaled.round(4)}")
print()

# Step 3: Causal mask
mask = np.triu(np.full((seq_len, seq_len), -np.inf), k=1)
scores_masked = scores_scaled + mask
print(f"Step 3 — Causal mask (upper triangle = -inf):")
print(f"  Mask:\n{mask}")
print(f"  Masked scores:")
for i, token in enumerate([decode([tid]) for tid in token_ids]):
    row = [f"{v:>8.4f}" if not np.isinf(v) else "    -inf" for v in scores_masked[i]]
    print(f"    {token:>6s} -> [{', '.join(row)}]")
print()
print(f"  'The' can only see itself.")
print(f"  ' cat' can see 'The' and itself.")
print(f"  ' sat' can see all three.")

In [ ]:
# Step 4: Softmax — convert to probabilities
attn_weights = softmax(scores_masked)  # (3, 3)

print(f"Step 4 — Attention weights (softmax):")
print(f"  Each row sums to 1.0 — like a load balancer distributing 100% of traffic.")
print()

token_strs = [decode([tid]) for tid in token_ids]
# Header
header = "              " + "".join(f"{t:>10s}" for t in token_strs)
print(header)
print("              " + "-" * (10 * len(token_strs)))
for i, src in enumerate(token_strs):
    row = "".join(f"{attn_weights[i][j]:>10.4f}" for j in range(seq_len))
    total = attn_weights[i].sum()
    print(f"  {src:>10s} -> {row}  (sum={total:.4f})")

print()
print("  'The' puts 100% attention on itself (only token it can see).")
print("  ' sat' distributes attention across all visible tokens.")

In [ ]:
# Step 5: Weighted sum of values
attn_output = attn_weights @ V  # (3, 768)

print(f"Step 5 — Attention output (weights x V):")
print(f"  Shape: {attn_output.shape} — same as input")
print()
print(f"  Each row is now a BLEND of the value vectors,")
print(f"  weighted by how much attention that token paid to each other token.")
print()
print(f"  Before attention: each token only knew about itself.")
print(f"  After attention:  each token has info from all tokens it attended to.")

## Step 3: Multi-Head Attention

Running attention once on the full 768-dim vectors gives one "perspective" on token relationships. But language has many kinds of relationships simultaneously:

- Subject-verb agreement: "The **cat** ... **sat**"
- Adjective-noun: "**big** ... **cat**"
- Coreference: "The cat ... **it** ... sat"

Multi-head attention runs **12 parallel attention operations**, each on a 64-dim slice:

```
768-dim vector
├── Head 0:  dims [0:64]     — maybe learns subject-verb
├── Head 1:  dims [64:128]   — maybe learns adjective-noun
├── Head 2:  dims [128:192]  — maybe learns coreference
├── ...
└── Head 11: dims [704:768]  — maybe learns something else

12 heads x 64 dims = 768 dims (no parameters added, just sliced differently)
```

It's like having 12 support agents each reading the same ticket but focusing on different aspects.

In [ ]:
n_heads = 12
head_dim = d_model // n_heads  # 768 // 12 = 64

print(f"Multi-head attention config:")
print(f"  n_heads:  {n_heads}")
print(f"  head_dim: {head_dim}  ({d_model} / {n_heads})")
print()

# Split Q, K, V into heads
# (seq_len, 768) -> (seq_len, 12, 64) -> (12, seq_len, 64)
Q_heads = Q.reshape(seq_len, n_heads, head_dim).transpose(1, 0, 2)
K_heads = K.reshape(seq_len, n_heads, head_dim).transpose(1, 0, 2)
V_heads = V.reshape(seq_len, n_heads, head_dim).transpose(1, 0, 2)

print(f"Q reshaped: {Q.shape} -> {Q_heads.shape}  (n_heads, seq_len, head_dim)")
print()

# Run attention for each head
mask = np.triu(np.full((seq_len, seq_len), -np.inf), k=1)

all_head_outputs = []
for h in range(n_heads):
    scores_h = Q_heads[h] @ K_heads[h].T / np.sqrt(head_dim)
    scores_h = scores_h + mask
    weights_h = softmax(scores_h)
    output_h = weights_h @ V_heads[h]  # (seq_len, 64)
    all_head_outputs.append(output_h)

# Show what different heads attend to for the last token (" sat")
print(f"Attention patterns for ' sat' (last token, can see all 3):")
print(f"{'Head':<6s} {'The':>8s} {' cat':>8s} {' sat':>8s}  Focus")
print("-" * 50)
for h in range(n_heads):
    scores_h = Q_heads[h] @ K_heads[h].T / np.sqrt(head_dim) + mask
    w = softmax(scores_h)
    row = w[-1]  # last token's attention
    focus_idx = np.argmax(row)
    focus_token = token_strs[focus_idx]
    print(f"  {h:<4d} {row[0]:>8.4f} {row[1]:>8.4f} {row[2]:>8.4f}  -> {focus_token}")

print()
print("Different heads focus on different tokens — that's the point.")

In [ ]:
# Concatenate all heads back together
# (12, seq_len, 64) -> (seq_len, 12, 64) -> (seq_len, 768)
multi_head_out = np.concatenate(all_head_outputs, axis=-1)  # (seq_len, 768)

# Output projection: one more matmul to mix information across heads
c_proj_W = weights["h.0.attn.c_proj.weight"]  # (768, 768)
c_proj_b = weights["h.0.attn.c_proj.bias"]    # (768,)

attn_final = multi_head_out @ c_proj_W + c_proj_b  # (seq_len, 768)

print(f"Concatenated heads: {multi_head_out.shape}  (12 heads of 64 = 768)")
print(f"Output projection:  {attn_final.shape}     (mixes info across heads)")
print()
print(f"The full attention pipeline:")
print(f"  Input:  (3, 768) — 3 token vectors")
print(f"  QKV:    one matmul -> Q, K, V")
print(f"  Split:  12 heads x 64 dims")
print(f"  Attend: each head computes its own attention")
print(f"  Concat: combine all heads")
print(f"  Proj:   one more matmul to mix across heads")
print(f"  Output: (3, 768) — same shape as input")

## Verify: Attention Weights Are Valid Probability Distributions

Every row of attention weights should:
- Sum to 1.0 (it's a probability distribution)
- Have all non-negative values
- Have zeros in the future positions (causal mask)

In [ ]:
# Check for head 0
scores_h0 = Q_heads[0] @ K_heads[0].T / np.sqrt(head_dim) + mask
w_h0 = softmax(scores_h0)

print("Verification for head 0:")
print(f"  All values >= 0:  {np.all(w_h0 >= 0)}")
print(f"  Row sums = 1.0:   {np.allclose(w_h0.sum(axis=-1), 1.0)}")
print(f"  Row sums:         {w_h0.sum(axis=-1).round(6)}")
print(f"  Upper triangle=0: {np.allclose(w_h0[0, 1:], 0)}")
print(f"    'The' attends to future tokens? {w_h0[0, 1] > 1e-10}  (should be False)")

## Why Scaling Matters

We divided scores by $\sqrt{d_k}$ = $\sqrt{64}$ = 8. Why?

Without scaling, the dot products grow with dimension size. Large values cause softmax to saturate — producing distributions like `[0.99, 0.01, 0.00]` instead of `[0.5, 0.3, 0.2]`.

This is similar to **normalizing metrics before comparing them** — you wouldn't compare "requests per second" (thousands) directly with "latency in ms" (single digits) without scaling first.

In [ ]:
# Demonstrate the effect of scaling
raw_scores = Q_heads[0] @ K_heads[0].T  # unscaled
scaled_scores = raw_scores / np.sqrt(head_dim)  # scaled

print("Last token's scores (before softmax):")
print(f"  Unscaled: {raw_scores[-1].round(2)}")
print(f"  Scaled:   {scaled_scores[-1].round(4)}")
print()

# Apply causal mask and softmax to both
raw_masked = raw_scores + mask
scaled_masked = scaled_scores + mask

w_unscaled = softmax(raw_masked)
w_scaled = softmax(scaled_masked)

print("Last token's attention weights (after softmax):")
print(f"  Unscaled: {w_unscaled[-1].round(6)}  <- one token dominates")
print(f"  Scaled:   {w_scaled[-1].round(6)}  <- more distributed")
print()
print("Without scaling, attention becomes nearly one-hot (winner takes all).")
print("With scaling, the model can attend to multiple tokens at once.")

## The Complete Attention Function

Let's package everything into one clean function — this is what runs 12 times per transformer block (once per layer, and GPT-2 has 12 layers = 144 total attention operations per forward pass).

In [ ]:
def multi_head_attention(x, c_attn_W, c_attn_b, c_proj_W, c_proj_b, n_heads):
    """Full multi-head self-attention using real GPT-2 weights.
    
    x:       (seq_len, d_model) — input embeddings
    returns: (seq_len, d_model) — attention output (same shape)
    """
    seq_len, d_model = x.shape
    head_dim = d_model // n_heads
    
    # Compute Q, K, V with one matmul
    qkv = x @ c_attn_W + c_attn_b        # (seq_len, 3*d_model)
    Q, K, V = np.split(qkv, 3, axis=-1)  # each (seq_len, d_model)
    
    # Split into heads
    Q = Q.reshape(seq_len, n_heads, head_dim).transpose(1, 0, 2)  # (n_heads, seq_len, head_dim)
    K = K.reshape(seq_len, n_heads, head_dim).transpose(1, 0, 2)
    V = V.reshape(seq_len, n_heads, head_dim).transpose(1, 0, 2)
    
    # Scaled dot-product attention
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(head_dim)  # (n_heads, seq_len, seq_len)
    mask = np.triu(np.full((seq_len, seq_len), -np.inf), k=1)
    scores = scores + mask
    attn_weights = softmax(scores)
    attn_out = attn_weights @ V  # (n_heads, seq_len, head_dim)
    
    # Concatenate heads and project
    attn_out = attn_out.transpose(1, 0, 2).reshape(seq_len, d_model)  # (seq_len, d_model)
    return attn_out @ c_proj_W + c_proj_b  # (seq_len, d_model)

# Test with real weights from block 0
text = "The cat sat on the mat"
token_ids = encode(text)
x = embed(token_ids)

attn_out = multi_head_attention(
    x,
    weights["h.0.attn.c_attn.weight"], weights["h.0.attn.c_attn.bias"],
    weights["h.0.attn.c_proj.weight"], weights["h.0.attn.c_proj.bias"],
    n_heads=12
)

print(f"Input:  {x.shape}")
print(f"Output: {attn_out.shape}")
print(f"\nThis ran real GPT-2 attention with real trained weights.")
print(f"No PyTorch. No transformers library. Just numpy matmuls.")

## Key Takeaways

- **Attention = database query.** Q asks, K indexes, V stores data. The result is a weighted blend of all values.
- **Causal mask** prevents looking at future tokens — the upper triangle of the score matrix is set to `-inf` before softmax.
- **Scaling by $1/\sqrt{d_k}$** prevents softmax saturation, keeping attention weights distributed rather than one-hot.
- **Multi-head** splits the 768-dim space into 12 heads of 64 dims each. Different heads learn different relationship types.
- **The full operation is 3 matmuls**: QKV projection, score computation (Q x K^T), value aggregation (weights x V), plus the output projection. That's it.
- GPT-2 runs this **12 times** (one per layer) = **144 attention operations** per forward pass.

## Next

**Day 03** — We add the FFN, layer norms, and residual connections to build a complete **transformer block**, then stack 12 of them for a full forward pass + text generation.

## References

- *Inference Engineering* Ch 2.2.3 (pp. 52–53) — Attention, self-attention, KV cache
- Vaswani et al., "Attention Is All You Need" (2017)
- Dao et al., "FlashAttention" (2022) — the optimized implementation used in production